# Hata Çubukları

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/04-matplotlib/03-errorbars.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 04.03 Errorbars

Her bilimsel ölçümde belirsizliklerin doğru hesaba katılması, çoğu zaman sayının kendisinin doğru raporlanması kadar — hatta daha da — önemlidir.
    Örneğin bazı astrofizik gözlemlerle Evren'in yerel genişleme hızı olan Hubble Sabiti'ni tahmin ettiğimi düşünün.
    Güncel literatür ~70 (km/s)/Mpc öneriyor; yöntemimle 74 (km/s)/Mpc ölçtüm. Değerler tutarlı mı? Verilen bilgiyle tek doğru cevap: bilemeyiz.

Bu bilgiye belirsizlikleri ekleyelim: literatür 70 ± 2,5 (km/s)/Mpc, yöntemim 74 ± 5 (km/s)/Mpc ölçtü.
    Şimdi tutarlılık nicel olarak sorulabilir.

Veri ve sonuçların görselleştirilmesinde bu hataların etkili gösterilmesi, grafiğin çok daha eksiksiz bilgi aktarmasını sağlar.

## Temel hata çubukları

Belirsizlikleri görselleştirmenin standart yollarından biri hata çubuğudur.
    Temel bir hata çubuğu tek bir Matplotlib fonksiyon çağrısıyla oluşturulabilir (aşağıdaki şekle bakın):


```python
# errorbars_setup.py
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import numpy as np
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


> **Not**
>


In [ ]:
# errorbar_basic.py
x = np.linspace(0, 10, 50)
dy = 0.8
y = np.sin(x) + dy * np.random.randn(50)

plt.errorbar(x, y, yerr=dy, fmt='.k');



Burada fmt, çizgi ve noktaların görünümünü kontrol eden bir biçim kodudur;
    önceki bölümde ve bu bölümün başında özetlenen plt.plot kısaltma sözdizimiyle aynıdır.

Temel seçeneklere ek olarak errorbar çıktıyı ince ayarlamak için birçok seçenek sunar.
    Özellikle kalabalık grafiklerde hata çubuklarını noktalardan daha açık renkte yapmayı yararlı bulurum (aşağıdaki şekle bakın):


In [ ]:
# errorbar_styled.py
plt.errorbar(x, y, yerr=dy, fmt='o', color='black',
             ecolor='lightgray', elinewidth=3, capsize=0);



### 🧪 Şimdi deneyin

🧪 Yatay hata çubukları
      Aynı veri için xerr ekleyerek yatay belirsizlik gösterin:
          
      import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 20)
y = np.sin(x)
dy = 0.15
dx = 0.3

plt.errorbar(x, y, xerr=dx, yerr=dy, fmt='o', color='black',
             ecolor='lightgray', capsize=3)
plt.title('Yatay ve dikey hata çubukları')

Bu seçeneklere ek olarak yatay hata çubukları, tek taraflı hata çubukları ve birçok varyant belirtilebilir.
    Kullanılabilir seçenekler için plt.errorbar docstring'ine bakın.

## Sürekli hatalar

Bazı durumlarda sürekli nicelikler üzerinde hata çubukları göstermek istenir.
    Matplotlib bu tür uygulama için hazır bir kolaylık rutini sunmasa da,
    plt.plot ve plt.fill_between gibi yapı taşlarını birleştirmek nispeten kolaydır.

Burada Scikit-Learn API'siyle basit bir Gaussian süreç regresyonu yapacağız.
    Bu, sürekli belirsizlik ölçüsüyle veriye çok esnek parametrik olmayan bir fonksiyon uydurma yöntemidir.
    Gaussian süreç regresyonunun ayrıntılarına girmeyeceğiz; bunun yerine bu tür sürekli hata ölçümünü nasıl görselleştirebileceğimize odaklanacağız:

> **Not**
>


In [ ]:
# gaussian_process_fit.py
from sklearn.gaussian_process import GaussianProcessRegressor

# define the model and draw some data
model = lambda x: x * np.sin(x)
xdata = np.array([1, 3, 5, 6, 8])
ydata = model(xdata)

# Compute the Gaussian process fit
gp = GaussianProcessRegressor()
gp.fit(xdata[:, np.newaxis], ydata)

xfit = np.linspace(0, 10, 1000)
yfit, dyfit = gp.predict(xfit[:, np.newaxis], return_std=True)



Artık verimize sürekli uydurmayı örnekleyen xfit, yfit ve dyfit değişkenlerimiz var.
    Bunları önceki bölümdeki gibi plt.errorbar'a verebilirdik; ancak 1.000 noktaya 1.000 hata çubuğu çizmek istemeyiz.
    Bunun yerine sürekli hatayı görselleştirmek için açık renkle plt.fill_between kullanabiliriz (aşağıdaki şekle bakın):


In [ ]:
# Visualize the result
plt.plot(xdata, ydata, 'or')
plt.plot(xfit, yfit, '-', color='gray')
plt.fill_between(xfit, yfit - dyfit, yfit + dyfit,
                 color='gray', alpha=0.2)
plt.xlim(0, 10);



fill_between çağrı imzasına bakın: bir x değeri, alt y sınırı, üst y sınırı verilir; bu bölgeler arası doldurulur.

Ortaya çıkan şekil, Gaussian süreç regresyon algoritmasının ne yaptığına sezgisel bir bakış verir:
    ölçülmüş veri noktasına yakın bölgelerde model güçlü kısıtlanır; model belirsizlikleri küçüktür.
    Uzak bölgelerde model zayıf kısıtlanır; belirsizlikler artar.

plt.fill_between (ve yakından ilişkili plt.fill) seçenekleri için fonksiyon docstring'ine veya Matplotlib dokümantasyonuna bakın.

Son olarak bu biraz düşük seviyeli geliyorsa,
    Seaborn ile Görselleştirme bölümüne bakın;
    Seaborn bu tür sürekli hata çubuklarını görselleştirmek için daha akıcı bir API sunar.

> **Not**
>
